# ThreatSense ML Training Notebook

This notebook provides an end-to-end workflow for training and evaluating machine learning models in Python.

Target workflow:
- Load and inspect dataset
- Clean and preprocess data
- Train baseline and tuned models
- Evaluate metrics and plots
- Save and reload the trained pipeline

## 1. Install and Import ML Libraries

Install packages only if your notebook kernel does not already have them.

In [1]:
# Uncomment if needed in a fresh kernel
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use('seaborn-v0_8')

## 2. Load Dataset

Place your NSL-KDD files in ml/data/raw:
- KDDTrain+.txt
- KDDTest+.txt

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    PROJECT_ROOT = ROOT.parent.parent
else:
    PROJECT_ROOT = ROOT

RAW_DIR = PROJECT_ROOT / 'ml' / 'data' / 'raw'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'models'
REPORT_DIR = PROJECT_ROOT / 'ml' / 'reports'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

train_path = RAW_DIR / 'KDDTrain+.txt'
test_path = RAW_DIR / 'KDDTest+.txt'

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        f'Missing dataset files in {RAW_DIR}. Add KDDTrain+.txt and KDDTest+.txt first.'
    )

column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty'
]

df_train = pd.read_csv(train_path, names=column_names)
df_test = pd.read_csv(test_path, names=column_names)

print('Train shape:', df_train.shape)
print('Test shape:', df_test.shape)
df_train.head()

## 3. Explore and Clean Data

In [3]:
print('Train info:')
print(df_train.info())

print('\nMissing values in train (top 10):')
print(df_train.isna().sum().sort_values(ascending=False).head(10))

before_dups = len(df_train)
df_train = df_train.drop_duplicates().reset_index(drop=True)
print(f'\nDropped duplicates: {before_dups - len(df_train)}')

class_counts = df_train['label'].value_counts().head(20)
plt.figure(figsize=(12, 5))
class_counts.plot(kind='bar')
plt.title('Top 20 Raw Labels in Train Set')
plt.xlabel('Raw label')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Encode Features and Scale Inputs

In [4]:
attack_map = {
    'normal': 'normal',
    'back': 'dos', 'land': 'dos', 'neptune': 'dos', 'pod': 'dos', 'smurf': 'dos', 'teardrop': 'dos',
    'mailbomb': 'dos', 'apache2': 'dos', 'processtable': 'dos', 'udpstorm': 'dos', 'worm': 'dos',
    'ipsweep': 'probe', 'nmap': 'probe', 'portsweep': 'probe', 'satan': 'probe', 'mscan': 'probe', 'saint': 'probe',
    'ftp_write': 'r2l', 'guess_passwd': 'r2l', 'imap': 'r2l', 'multihop': 'r2l', 'phf': 'r2l',
    'spy': 'r2l', 'warezclient': 'r2l', 'warezmaster': 'r2l', 'sendmail': 'r2l', 'named': 'r2l',
    'snmpgetattack': 'r2l', 'snmpguess': 'r2l', 'xlock': 'r2l', 'xsnoop': 'r2l', 'httptunnel': 'r2l',
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r', 'perl': 'u2r', 'rootkit': 'u2r',
    'ps': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r'
}

def map_attack(label: str) -> str:
    base = str(label).strip().rstrip('.')
    return attack_map.get(base, 'probe')

for frame in (df_train, df_test):
    frame['attack_class'] = frame['label'].apply(map_attack)

feature_cols = [c for c in df_train.columns if c not in ['label', 'difficulty', 'attack_class']]
categorical_cols = ['protocol_type', 'service', 'flag']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X_train_full = df_train[feature_cols]
y_train_full = df_train['attack_class']
X_test_holdout = df_test[feature_cols]
y_test_holdout = df_test['attack_class']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_cols,
        ),
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ),
    ]
)

print('Categorical columns:', categorical_cols)
print('Numeric columns:', len(numeric_cols))
print(y_train_full.value_counts())

## 5. Split Data into Train, Validation, and Test Sets

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print('X_train:', X_train.shape)
print('X_val:', X_val.shape)
print('X_test_holdout:', X_test_holdout.shape)

## 6. Train a Baseline Model

In [6]:
baseline_pipeline = Pipeline([
    ('preprocess', preprocessor),
    (
        'model',
        RandomForestClassifier(
            n_estimators=150,
            max_depth=None,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    ),
])

baseline_pipeline.fit(X_train, y_train)
y_val_pred_base = baseline_pipeline.predict(X_val)

base_acc = accuracy_score(y_val, y_val_pred_base)
print(f'Baseline validation accuracy: {base_acc:.4f}')

## 7. Evaluate Model Performance

In [7]:
labels_sorted = sorted(y_val.unique())

print('Baseline classification report:')
print(classification_report(y_val, y_val_pred_base, digits=4))

cm = confusion_matrix(y_val, y_val_pred_base, labels=labels_sorted)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_sorted, yticklabels=labels_sorted)
plt.title('Baseline Validation Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 8. Tune Hyperparameters

In [8]:
search_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE)),
])

param_dist = {
    'model__n_estimators': [120, 180, 250],
    'model__max_depth': [None, 20, 35],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
}

search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_dist,
    n_iter=8,
    scoring='f1_weighted',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV score:', round(search.best_score_, 4))

## 9. Train Final Model and Compare Metrics

In [9]:
final_model = search.best_estimator_

# Evaluate baseline and tuned models on holdout test set
y_test_pred_base = baseline_pipeline.predict(X_test_holdout)
y_test_pred_final = final_model.predict(X_test_holdout)

base_precision, base_recall, base_f1, _ = precision_recall_fscore_support(
    y_test_holdout, y_test_pred_base, average='weighted', zero_division=0
)
final_precision, final_recall, final_f1, _ = precision_recall_fscore_support(
    y_test_holdout, y_test_pred_final, average='weighted', zero_division=0
)

comparison = pd.DataFrame([
    {
        'model': 'baseline_random_forest',
        'accuracy': accuracy_score(y_test_holdout, y_test_pred_base),
        'precision_weighted': base_precision,
        'recall_weighted': base_recall,
        'f1_weighted': base_f1,
    },
    {
        'model': 'tuned_random_forest',
        'accuracy': accuracy_score(y_test_holdout, y_test_pred_final),
        'precision_weighted': final_precision,
        'recall_weighted': final_recall,
        'f1_weighted': final_f1,
    },
])

display(comparison)

print('Final classification report on holdout test:')
print(classification_report(y_test_holdout, y_test_pred_final, digits=4))

## 10. Save and Reload the Trained Model

In [10]:
model_path = MODEL_DIR / 'trained_pipeline.joblib'
preprocess_path = MODEL_DIR / 'preprocess_bundle.joblib'
metrics_path = REPORT_DIR / 'metrics.json'

joblib.dump(final_model, model_path)
joblib.dump({'feature_columns': feature_cols, 'categorical_cols': categorical_cols, 'numeric_cols': numeric_cols}, preprocess_path)

loaded_model = joblib.load(model_path)
sample_pred = loaded_model.predict(X_test_holdout.head(3))
print('Sample predictions:', sample_pred)

metrics_payload = {
    'baseline': comparison.iloc[0].to_dict(),
    'tuned': comparison.iloc[1].to_dict(),
    'best_params': search.best_params_,
}

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

print('Saved model to:', model_path)
print('Saved preprocess metadata to:', preprocess_path)
print('Saved metrics to:', metrics_path)

## 11. ThreatSense Extension: ANN and Isolation Forest (Optional)

Use this section when you are ready to move from baseline scikit-learn models to your project target stack.

In [11]:
from sklearn.ensemble import IsolationForest

# Isolation Forest on transformed feature space
X_train_trans = preprocessor.fit_transform(X_train)
iforest = IsolationForest(n_estimators=200, contamination=0.08, random_state=RANDOM_STATE)
iforest.fit(X_train_trans)

X_test_trans = preprocessor.transform(X_test_holdout)
anomaly_flag = iforest.predict(X_test_trans)  # -1 anomaly, 1 normal
anomaly_share = (anomaly_flag == -1).mean()
print(f'Anomaly share on holdout: {anomaly_share:.4f}')

iforest_path = MODEL_DIR / 'iforest.joblib'
joblib.dump(iforest, iforest_path)
print('Saved Isolation Forest to:', iforest_path)

# Optional ANN training block
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.utils import to_categorical

    classes = sorted(y_train.unique())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    y_train_idx = y_train.map(class_to_idx).values
    y_val_idx = y_val.map(class_to_idx).values

    X_train_arr = preprocessor.fit_transform(X_train)
    X_val_arr = preprocessor.transform(X_val)

    # Sparse matrix may be returned when one-hot encoding is used.
    if hasattr(X_train_arr, 'toarray'):
        X_train_arr = X_train_arr.toarray()
        X_val_arr = X_val_arr.toarray()

    ann = Sequential([
        Dense(128, activation='relu', input_shape=(X_train_arr.shape[1],)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(len(classes), activation='softmax'),
    ])

    ann.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    history = ann.fit(
        X_train_arr,
        to_categorical(y_train_idx),
        validation_data=(X_val_arr, to_categorical(y_val_idx)),
        epochs=8,
        batch_size=256,
        verbose=1,
    )

    ann_path = MODEL_DIR / 'ann_model.h5'
    ann.save(ann_path)
    print('Saved ANN to:', ann_path)

    plt.figure(figsize=(8, 4))
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.title('ANN Accuracy by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print('ANN block skipped:', exc)

In [16]:
# 6. Model Comparison Summary
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

comparison_data = {
    'Model': ['RandomForest Classifier', 'ANN (Keras)', 'Isolation Forest'],
    'Accuracy': [f"{rf_accuracy:.4f}", f"{ann_accuracy:.4f}", "N/A"],
    'Precision': [f"{rf_precision:.4f}", f"{ann_precision:.4f}", "N/A"],
    'Recall': [f"{rf_recall:.4f}", f"{ann_recall:.4f}", "N/A"],
    'F1-Score': [f"{rf_f1:.4f}", f"{ann_f1:.4f}", "N/A"],
    'Type': ['Classification', 'Classification', 'Anomaly Detection']
}

import pandas as pd
comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Determine best classifier
if ann_accuracy > rf_accuracy:
    best_model = f"ANN ({ann_accuracy:.2%} accuracy)"
    best_f1 = ann_f1
else:
    best_model = f"RandomForest ({rf_accuracy:.2%} accuracy)"
    best_f1 = rf_f1

print("\n" + "=" * 80)
print(f"✓ BEST CLASSIFIER: {best_model}")
print(f"✓ BEST F1-SCORE:   {best_f1:.4f}")
print(f"✓ ANOMALY DETECTION: Isolation Forest ({anomaly_share*100:.2f}% anomalies detected)")
print("=" * 80)

In [15]:
# 5. Confusion Matrices Visualization
print("\n5. CONFUSION MATRICES")
print("-" * 80)

cm_rf = confusion_matrix(y_test_holdout, rf_predictions)
cm_ann = confusion_matrix(y_test_holdout, y_pred_ann)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RandomForest Confusion Matrix
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes, ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title(f'RandomForest Confusion Matrix\n(Accuracy: {rf_accuracy:.2%})')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ANN Confusion Matrix
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Greens', 
            xticklabels=classes, yticklabels=classes, ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title(f'ANN Confusion Matrix\n(Accuracy: {ann_accuracy:.2%})')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("✓ Confusion matrices plotted")

In [14]:
# 4. Detailed Classification Reports
print("\n4. DETAILED CLASSIFICATION REPORTS")
print("-" * 80)

print("\nRANDOMFOREST CLASSIFIER:")
print(classification_report(y_test_holdout, rf_predictions, zero_division=0))

print("\nKERAS ANN MODEL:")
print(classification_report(y_test_holdout, y_pred_ann, zero_division=0))

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("=" * 80)
print("COMPREHENSIVE MODEL EVALUATION")
print("=" * 80)

# 1. Evaluate RandomForest on test set
print("\n1. RANDOMFOREST CLASSIFIER EVALUATION")
print("-" * 80)
rf_predictions = final_model.predict(X_test_holdout)
rf_accuracy = (rf_predictions == y_test_holdout).mean()
rf_precision = precision_score(y_test_holdout, rf_predictions, average='weighted', zero_division=0)
rf_recall = recall_score(y_test_holdout, rf_predictions, average='weighted', zero_division=0)
rf_f1 = f1_score(y_test_holdout, rf_predictions, average='weighted', zero_division=0)

print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-Score:  {rf_f1:.4f}")

# 2. Evaluate ANN on test set
print("\n2. KERAS ANN MODEL EVALUATION")
print("-" * 80)

# Prepare test data for ANN
X_test_trans_ann = preprocessor.transform(X_test_holdout)
if hasattr(X_test_trans_ann, 'toarray'):
    X_test_trans_ann = X_test_trans_ann.toarray()

# Get ANN predictions
y_pred_proba_ann = ann.predict(X_test_trans_ann, verbose=0)
y_pred_ann_idx = np.argmax(y_pred_proba_ann, axis=1)

# Convert indices back to class labels
idx_to_class = {i: c for i, c in enumerate(classes)}
y_pred_ann = np.array([idx_to_class[i] for i in y_pred_ann_idx])

ann_accuracy = (y_pred_ann == y_test_holdout).mean()
ann_precision = precision_score(y_test_holdout, y_pred_ann, average='weighted', zero_division=0)
ann_recall = recall_score(y_test_holdout, y_pred_ann, average='weighted', zero_division=0)
ann_f1 = f1_score(y_test_holdout, y_pred_ann, average='weighted', zero_division=0)

print(f"Accuracy:  {ann_accuracy:.4f}")
print(f"Precision: {ann_precision:.4f}")
print(f"Recall:    {ann_recall:.4f}")
print(f"F1-Score:  {ann_f1:.4f}")

# 3. Isolation Forest Anomaly Detection
print("\n3. ISOLATION FOREST - ANOMALY DETECTION")
print("-" * 80)
print(f"Anomaly Share: {anomaly_share:.4f} ({anomaly_share*100:.2f}%)")
print(f"Normal Share:  {1-anomaly_share:.4f} ({(1-anomaly_share)*100:.2f}%)")